## Setup runner & utilities

In [1]:
from nanover.app import OmniRunner
from nanover.openmm import OpenMMSimulation

simulation = OpenMMSimulation.from_xml_path("../openmm/openmm_files/17-ala.xml")
simulation.load()

imd_runner = OmniRunner.with_basic_server(simulation, port=0, name="EXAMPLE: moveable restraints")
imd_runner.load(0)

In [2]:
from nanover.jupyter import NanoverJupyterUtilities

utilities = NanoverJupyterUtilities.from_runner(imd_runner)

In [3]:
utilities.use_recording_commands()
utilities.use_interaction_modes()
utilities.selections.update_selection("root", renderer="cartoon")

## Restraints

In [4]:
from nanover.imd.imd_state import ParticleInteraction

RESTRAINT_PREFIX = f"MOVEABLE-RESTRAINT"

NEXT_RESTRAINT_INDEX = 0
MOVEABLE_RESTRAINTS: dict[str, ParticleInteraction] = {}
ACTIVE_RESTRAINTS: set[str] = set()

RESTRAINT_SELECTION_KWARGS = dict(
    renderer={
        "render": {
            "particle.scale": .1,
            "bond.scale": 0.0,
            "type": "ball and stick"
        },
        "color": "CornflowerBlue",
    },
    interaction_method="group",
    velocity_reset=True,
)

def add_restraint(indexes):
    global NEXT_RESTRAINT_INDEX
    indexes = [int(index) for index in indexes]

    key = f"{RESTRAINT_PREFIX}.{NEXT_RESTRAINT_INDEX}"
    NEXT_RESTRAINT_INDEX += 1
    restraint = ParticleInteraction((0, 0, 0), indexes, scale=1000, interaction_type="spring")
    MOVEABLE_RESTRAINTS[key] = restraint

    utilities.notify_all("ADD RESTRAINT")

    utilities.selections.update_selection(
        key,
        particle_ids=indexes,
        **RESTRAINT_SELECTION_KWARGS,
    )

    enable_restraint(key)

    return key


def remove_restraint(key: str):
    disable_restraint(key)
    utilities.selections.remove_selection(key)
    MOVEABLE_RESTRAINTS.pop(key, None)
    utilities.objects.remove_shape(key)
    utilities.objects.remove_line(key)


def enable_restraint(key: str):
    if key in ACTIVE_RESTRAINTS:
        return
    ACTIVE_RESTRAINTS.add(key)
    positions = imd_runner.app_server.frame_publisher.current_frame.particle_positions
    restraint = MOVEABLE_RESTRAINTS[key]
    restraint.position = np.mean(positions[restraint.particles], axis=0)
    utilities.notify_all("ENABLE RESTRAINT")
    utilities.interactions.update_interaction(key, restraint)


def disable_restraint(key: str):
    if key not in ACTIVE_RESTRAINTS:
        return
    ACTIVE_RESTRAINTS.remove(key)
    utilities.interactions.remove_interaction(key)

## Visuals

In [5]:
import numpy as np
from nanover.trajectory import FrameData
from nanover.jupyter import FrameListener

# TODO: why does unsanitised state update break the frame stream but only for this thread?

class MobileRestraintVisuals(FrameListener):
    def on_frame_update(self, full_frame: FrameData, frame_update: FrameData):
        with utilities.objects as objects:
            for key, restraint in MOVEABLE_RESTRAINTS.items():
                centroid = np.mean(full_frame.particle_positions[restraint.particles], axis=0)
                distance = np.linalg.norm(centroid - restraint.position)
                strain = min(distance / .5, 1.0)
                color = [1.0, 1.0 - strain, 1.0 - strain, 1.0]
                objects.update_shape(key, position=restraint.position, scale=.5, color=color)
                objects.update_line(key, positions=[restraint.position, centroid], scale=.1, color=color)


visuals = MobileRestraintVisuals.from_runner(imd_runner)
visuals.start()

## Commands

In [6]:
def clear_restraints():
    for restraint in list(MOVEABLE_RESTRAINTS.keys()):
        remove_restraint(restraint)
    MOVEABLE_RESTRAINTS.clear()

imd_runner.app_server.register_command("user/restraints/clear", clear_restraints, label="clear restraints", icon="🚫")

In [7]:
from nanover.jupyter import Mode

class InteractMode(Mode):
    def on_interaction_started(self, *, key: str, interaction: ParticleInteraction):
        if RESTRAINT_PREFIX in key:
            return

        interacted_indexes = set(interaction.particles)
        for key, restraint in MOVEABLE_RESTRAINTS.items():
            if interacted_indexes.intersection(restraint.particles):
                disable_restraint(key)

    def on_interaction_stopped(self, *, key: str, interaction: ParticleInteraction):
        if RESTRAINT_PREFIX in key:
            return

        interacted_indexes = set(interaction.particles)
        for key, restraint in MOVEABLE_RESTRAINTS.items():
            if interacted_indexes.intersection(restraint.particles):
                enable_restraint(key)

utilities.add_interaction_mode(InteractMode, "normal")

In [8]:
from nanover.jupyter import Mode

class ToggleMode(Mode):
    def on_interaction_started(self, *, key: str, interaction: ParticleInteraction):
        if RESTRAINT_PREFIX in key:
            return

        interacted_indexes = set(interaction.particles)
        removed = False
        for key, restraint in MOVEABLE_RESTRAINTS.items():
            if interacted_indexes.intersection(restraint.particles):
                remove_restraint(key)
                removed = True
        if not removed:
            add_restraint(interaction.particles)

utilities.add_interaction_mode(ToggleMode, "toggle", icon="⛓️")